# Week 3: Leakage Audit
Ensuring no future information is leaked during early detection window framing.

In [ ]:
import pandas as pd
import numpy as np

# Load Data
data_path = '../data/processed/phase2_ingestion/unified.parquet'
df = pd.read_parquet(data_path)
print(f"Loaded {len(df)} tweets.")

## 1. Temporal Leakage Check
Verify timestamps are correctly normalized ($t \ge 0$).

In [ ]:
invalid_timestamps = df[df['timestamp'] < 0]
print(f"Tweets with negative timestamps: {len(invalid_timestamps)}")
if len(invalid_timestamps) > 0:
    display(invalid_timestamps.head())
else:
    print("SUCCESS: All timestamps are >= 0.")

## 2. Observation Window Simulation
Simulate filtering the cascade at $T=1$ hour and ensure future nodes are dropped.

In [ ]:
def simulate_observation_window(cascade_df, window_hours=1):
    window_seconds = window_hours * 3600
    filtered_df = cascade_df[cascade_df['timestamp'] <= window_seconds]
    return filtered_df

sample_cascade = df[df['cascade_id'] == df['cascade_id'].iloc[0]]
print(f"Original size: {len(sample_cascade)}")

filtered_cascade = simulate_observation_window(sample_cascade, window_hours=1)
print(f"Size after 1-hour window: {len(filtered_cascade)}")

# Verify max timestamp
max_ts_hours = filtered_cascade['timestamp'].max() / 3600
print(f"Max timestamp in filtered (hours): {max_ts_hours:.2f}")
assert filtered_cascade['timestamp'].max() <= 3600, "Leakage detected!"
print("SUCCESS: Observation window filter successfully drops future nodes.")

## 3. Feature Leakage Framework
Refer to `docs/validated_features.md` for the formal Feature Leakage Table.